# 02. Find the climate posts

Reads the raw vendor corpus and pulls out the climate-related posts for 2022, together
with everything the analysis and the referees need: the author, the language, the
client app, the engagement counts, and per-country denominators over *all* posts.

## What changed in this revision

The vendor delivers Twitter v1.1 streaming JSON. For any post longer than 140
characters the `text` field is cut short and the rest sits in
`extended_tweet.full_text`. The 2025 run read only `text`, so **61% of the posts it
analysed were truncated**, and unevenly across countries (Egypt 76%, Pakistan 73%,
United States 57%). Worse, a post whose only climate keyword fell after character 140
was never picked up at all.

This notebook runs the same Global Pulse filter on the **full** text. Section 2
measures what that changes on a single day before committing to the full corpus, and
section 5 reports it exactly for the whole year.

## Cost

About 343M posts. Measured on an M1 Pro: 57k posts/s per core for the filter, and
roughly 18k posts/s per core once JSON parsing is included, so **45-60 minutes on 7
workers**. Output is around 1 GB.

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))  # the modules the notebooks import live in src/


In [ ]:
import os
import glob

import numpy as np
import pandas as pd

import config as cfg
import climate_filter
import tweets_filter as tf

RAW_DIR = cfg.RAW_TWEET_DIR
OUT_DIR = os.path.join(cfg.OUTPUT_DIR, 'tweets_2022')

# Section 3 reads the whole corpus. Leave this False until section 2 looks right.
RUN_FULL = False

print("Raw files :", len(glob.glob(os.path.join(RAW_DIR, 'Global-EC-*.json.gz'))))
print("Output    :", OUT_DIR)

## 1. The Global Pulse filter

`climate_filter.py` is lifted verbatim from the 2025 notebook: the same keyword
lists, the same Boolean structure, the same negative terms. Keeping it byte-identical
is the point — it means any difference between the old sample and the new one is
caused by the text the filter reads, not by the filter itself.

The taxonomy is a set of OR-ed rules. Some are plain keyword lists ("climate change",
"global warming", "fossil fuel"); others are conjunctions that need a climate word
*and* a topical word, which is what keeps "drought" in a football report out.

In [ ]:
### Spot-check the filter on posts of each kind

examples = [
    ("carbon tax spin was a real conservative tactic win", True),
    ("#cop27 was a disappointment", True),
    ("global warming is a hoax", True),
    ("the environment ministry announced new flood defences", True),
    ("scored two goals in the second half despite the drought", False),
    ("just had a great sandwich", False),
]
for txt, expected in examples:
    got = climate_filter.is_climate_tweet(txt.lower())
    print(f"  {str(got):<5s} (expected {str(expected):<5s})  {txt}")

## 2. What the full text changes, on one day

2022-06-15 is a good test day: notebook 04's recovery check showed the four raw files
for that day span 00:02 to 19:47, and the 2025 analysis set has **952** posts inside
that window.

So filtering on `text` should land near 952. That is the check that the filter is
faithful. Filtering on `full_text` then shows the honest increment.

In [ ]:
import gzip
import json as _json

day_files = sorted(glob.glob(os.path.join(RAW_DIR, 'Global-EC-2022-06-15-*.json.gz')))
print("Files:", [os.path.basename(f) for f in day_files])

n = n_geo = n_trunc = 0
hit_text = hit_full = both = only_full = only_text = 0

for path in day_files:
    with gzip.open(path, 'rb') as fh:
        for line in fh:
            try:
                d = _json.loads(line)
            except Exception:
                continue
            n += 1
            place = d.get('place') or {}
            if not place:
                continue
            n_geo += 1
            ext = d.get('extended_tweet')
            text = d.get('text', '')
            full = (ext or {}).get('full_text') or text
            if ext:
                n_trunc += 1
            a = climate_filter.is_climate_tweet(text.lower())
            b = climate_filter.is_climate_tweet(full.lower())
            hit_text += a
            hit_full += b
            both += (a and b)
            only_full += (b and not a)
            only_text += (a and not b)

print(f"\nposts {n:,}   geotagged {n_geo:,} ({100*n_geo/n:.1f}%)   truncated {n_trunc:,} ({100*n_trunc/n:.1f}%)")
print(f"climate on short text : {hit_text:,}   (the 2025 run found 952 in this window)")
print(f"climate on full text  : {hit_full:,}")
print(f"  found by both       : {both:,}")
print(f"  only with full text : {only_full:,}   <- missed in 2025")
print(f"  only with short text: {only_text:,}   <- lost when the tail is read")
print(f"\nnet change: {100*(hit_full-hit_text)/max(hit_text,1):+.1f}%")

In [ ]:
### Look at what the short text was hiding

shown = 0
for path in day_files:
    if shown >= 4:
        break
    with gzip.open(path, 'rb') as fh:
        for line in fh:
            try:
                d = _json.loads(line)
            except Exception:
                continue
            ext = d.get('extended_tweet')
            if not ext or not (d.get('place') or {}):
                continue
            text, full = d.get('text', ''), ext.get('full_text', '')
            if climate_filter.is_climate_tweet(full.lower()) and \
               not climate_filter.is_climate_tweet(text.lower()):
                print(f"  short ({len(text):3d}): {text[:130]}")
                print(f"  full  ({len(full):3d}): {full[:260]}")
                print()
                shown += 1
                if shown >= 4:
                    break

## 3. Run over the corpus

`tweets_filter.process` does everything in one pass per file: the filter on the full
text, the filter on the short text wherever the post was truncated, the per-country
denominators, and a HyperLogLog sketch of distinct authors.

Restartable: finished files are appended to `done.log` and skipped on a rerun.

In [ ]:
if RUN_FULL:
    tf.run_parallel(RAW_DIR, OUT_DIR, workers=7, year=cfg.ANALYSIS_YEAR)
else:
    print("RUN_FULL is False. Set it to True in section 0 to scan the corpus.")

## 4. Assemble the sample

Posts are deduplicated on `tweet_id`. The 2025 run deduplicated on text and lost 61%
of its rows, because the vendor ships four overlapping products (`Global-EC`,
`Global-UC-EC`, `US-EC`, `US-UC-EC`) and the old input folder mixed them. Only
`Global-EC` is read here, so duplicates should be rare.

In [ ]:
parts = sorted(glob.glob(os.path.join(OUT_DIR, '*.tweets.csv')))
print("Part files:", len(parts))

posts = pd.concat([pd.read_csv(f, low_memory=False) for f in parts], ignore_index=True)
before = len(posts)
posts = posts.drop_duplicates(subset='tweet_id')

for c in ['was_truncated', 'hit_on_text', 'is_rt', 'is_quote', 'is_reply', 'has_coords']:
    posts[c] = pd.to_numeric(posts[c], errors='coerce')
for c in ['verified', 'default_profile', 'default_profile_image']:
    posts[c] = posts[c].astype(str).str.lower().map({'true': 1, 'false': 0})

posts.to_parquet(os.path.join(cfg.OUTPUT_DIR, 'climate_posts_2022.parquet'), index=False)
print(f"rows {before:,} -> {len(posts):,} after dropping duplicate ids "
      f"({before - len(posts):,} removed)")

In [ ]:
### The sample funnel, for R2.1

agg = pd.concat([pd.read_csv(f) for f in glob.glob(os.path.join(OUT_DIR, '*.agg.csv'))],
                ignore_index=True)

funnel = pd.DataFrame([
    ("Posts in the Global-EC feed, 2022",        int(agg.n.sum())),
    ("  of which geotagged",                     int(agg[agg.country_code.notna()].n.sum())),
    ("Match the Global Pulse taxonomy",          int(agg.n_climate.sum())),
    ("  would have matched on the short text",   int(agg.n_climate_on_text.sum())),
    ("Geotagged and 2022 (analysis sample)",     len(posts)),
    ("  distinct authors",                       posts.user_id.nunique()),
], columns=['step', 'n'])
funnel['share'] = (100 * funnel.n / funnel.n.iloc[0]).round(3)
funnel.to_csv(os.path.join(cfg.OUTPUT_DIR, 'sample_funnel.csv'), index=False)
funnel

## 5. What the truncation cost

`hit_on_text` records, for every post kept, whether the 2025 run would have found it.
The posts where it is 0 are the ones truncation hid.

In [ ]:
missed = (posts.hit_on_text == 0)
print(f"posts kept                  : {len(posts):,}")
print(f"  truncated at 140 chars    : {int(posts.was_truncated.sum()):,} "
      f"({100*posts.was_truncated.mean():.1f}%)")
print(f"  invisible to the 2025 run : {int(missed.sum()):,} ({100*missed.mean():.1f}%)")
print(f"\nmean text length: short {posts.text.str.len().mean():.0f} chars, "
      f"full {posts.full_text.str.len().mean():.0f} chars")

by_country = (posts.groupby('country_code')
                   .agg(posts=('tweet_id', 'size'),
                        truncated=('was_truncated', 'mean'),
                        missed_in_2025=('hit_on_text', lambda s: 1 - s.mean()))
                   .sort_values('posts', ascending=False))
by_country[['truncated', 'missed_in_2025']] = \
    (100 * by_country[['truncated', 'missed_in_2025']]).round(1)
by_country.to_csv(os.path.join(cfg.OUTPUT_DIR, 'truncation_by_country.csv'))
by_country.head(15)

## 6. Per-country denominators

Counts over every post in the feed, not just the climate ones. These answer the
descriptive statistics both referees asked for: posts and distinct authors per
country (R2.1, R1.3.a), retweet share, and the language mix that explains why the
analysis is English only (R2.2).

In [ ]:
lang = agg.groupby(['country_code', 'lang'], dropna=False)['n'].sum().reset_index()
lang.to_csv(os.path.join(cfg.OUTPUT_DIR, 'lang_by_country.csv'), index=False)

tot = agg.groupby('country_code', dropna=False)[
    ['n', 'n_rt', 'n_reply', 'n_quote', 'n_truncated', 'n_verified', 'n_climate']].sum()

# distinct authors over all posts, merged from the per-file sketches
users = tf.merge_hll(glob.glob(os.path.join(OUT_DIR, '*.hll.pkl.gz')))
tot['unique_authors_est'] = pd.Series(users).reindex(tot.index)

en = lang[lang.lang == 'en'].set_index('country_code')['n']
tot['en_share'] = (en.reindex(tot.index) / tot.n).round(4)
for c in ['rt', 'reply', 'quote', 'truncated']:
    tot[c + '_share'] = (tot['n_' + c] / tot.n).round(4)

cm = posts.groupby('country_code')
tot['climate_posts'] = cm.size()
tot['climate_authors'] = cm.user_id.nunique()

tot.sort_values('n', ascending=False).to_csv(
    os.path.join(cfg.OUTPUT_DIR, 'country_totals.csv'))

g = lang.groupby('lang')['n'].sum().sort_values(ascending=False)
print("Language shares of the whole feed:")
print((100 * g / g.sum()).round(2).head(6).to_string())
tot.sort_values('n', ascending=False).head(12)[
    ['n', 'unique_authors_est', 'en_share', 'rt_share', 'climate_posts', 'climate_authors']]

## 7. How precise is the filter?

Reading the full text nearly doubles the sample, but not all of that is climate
content. The taxonomy's conjunction rules — a climate word plus a topical word —
fire more readily on long text simply because there is more text to match, so
"safer environment on Columbia Ave" and "Environmental Services Tech jobs" come in
alongside genuine posts. The short text was acting as an accidental precision filter.

Nobody has measured the taxonomy's precision on this corpus, in this paper or in the
2025 run. This section does that, which both settles whether the extra posts are
sound and gives the descriptive statistics section a number worth reporting.

The sample is stratified: half from the posts the 2025 run also found, half from the
ones only the full text finds. Rows are shuffled and the stratum is not shown, so the
labeller cannot be swayed by knowing which is which.

In [ ]:
### Draw a stratified sample to label

N_PER_STRATUM = 75
SAMPLE_PATH = os.path.join(cfg.OUTPUT_DIR, 'labelling_sample.csv')

rng = np.random.default_rng(cfg.SEED)
strata = {1: posts[posts.hit_on_text == 1], 0: posts[posts.hit_on_text == 0]}
picked = pd.concat([
    s.sample(min(N_PER_STRATUM, len(s)), random_state=cfg.SEED) for s in strata.values()
])

sample = picked[['tweet_id', 'country', 'full_text']].copy()
sample['rules'] = [', '.join(tf.fired_rules(t.lower())) for t in picked.full_text]
sample['is_climate'] = ''          # 1 = yes, 0 = no, ? = unsure
sample['_stratum'] = picked.hit_on_text.values   # kept for scoring, ignore when labelling

sample = sample.sample(frac=1, random_state=cfg.SEED).reset_index(drop=True)
sample.to_csv(SAMPLE_PATH, index=False)

print(f"Wrote {len(sample)} posts to {SAMPLE_PATH}")
print("Fill in `is_climate` with 1, 0 or ?, and ignore the `_stratum` column.")
sample.head(3)[['country', 'full_text', 'rules']]

### Score the labels

Run this once the sample has been labelled. Two labellers should each keep their own
copy; pass both to get an agreement rate alongside the precision estimates.

Precision here means: of the posts the filter selected, how many are genuinely about
climate. The two strata are then combined, weighted by how many posts each stratum
actually contributes, to give precision for the sample as a whole.

In [ ]:
LABELLED = [SAMPLE_PATH]        # add a second path if two people labelled separately

labels = [pd.read_csv(f) for f in LABELLED]
if any(l.is_climate.isna().all() for l in labels):
    print("Nothing labelled yet — fill in the `is_climate` column first.")
else:
    if len(labels) == 2:
        a, b = (l.sort_values('tweet_id').is_climate.astype(str) for l in labels)
        print(f"Agreement between labellers: {100*(a.values == b.values).mean():.1f}%")

    lab = labels[0]
    lab = lab[lab.is_climate.astype(str).isin(['0', '1'])].copy()
    lab['is_climate'] = lab.is_climate.astype(int)

    rows = []
    for stratum, name in [(1, 'found in 2025 too'), (0, 'only with full text')]:
        s = lab[lab._stratum == stratum]
        k, n = int(s.is_climate.sum()), len(s)
        lo, hi = tf.wilson_ci(k, n)
        rows.append((name, n, k, k / n, lo, hi, int((posts.hit_on_text == stratum).sum())))

    prec = pd.DataFrame(rows, columns=['stratum', 'labelled', 'climate', 'precision',
                                       'ci_low', 'ci_high', 'posts_in_sample'])
    # Weight the two strata by their real share of the sample
    w = prec.posts_in_sample / prec.posts_in_sample.sum()
    overall = float((prec.precision * w).sum())

    prec.to_csv(os.path.join(cfg.OUTPUT_DIR, 'filter_precision.csv'), index=False)
    print(prec.round(3).to_string(index=False))
    print(f"\nWeighted precision over the whole sample: {overall:.1%}")
    print(f"Implied climate posts after correction: {overall*len(posts):,.0f} of {len(posts):,}")

In [ ]:
### Precision by rule type, which is where the difference should show up

if 'lab' in dir() and len(lab):
    lab['strong'] = [any(r in tf.STRONG_RULES for r in tf.fired_rules(t.lower()))
                     for t in lab.full_text]
    by_rule = lab.groupby('strong').agg(labelled=('is_climate', 'size'),
                                        precision=('is_climate', 'mean'))
    by_rule.index = ['weak rules only', 'names climate directly']
    print(by_rule.round(3).to_string())
    print("\nIf the weak-rule posts are much less precise, say so in the Methods and "
          "report both numbers rather than quietly dropping them.")